# Shared Layered Network Generation

Generates layered-graph network instances used by **both** NFG and DFG experiments.

Each instance is saved as a JSON file (same format as `00_generate_networks.ipynb`)
so that NFG/DFG experiment notebooks can load them identically.

## Layered Graph Design

- **L** layers × **W** nodes per layer → L×W total nodes
- Random edges between adjacent layers with probability **p = 0.8**
- Connectivity guarantee: every layer-0 node can reach some layer-(L-1) node, and vice versa
- **Asymmetric only**: sources from layer 0, sinks from layer L-1

## Experimental Design

Grid: `n_layers × n_width × capacity_level × n_players × seed`

- `n_layers` ∈ {3, 5, 8, 10}, `n_width` ∈ {3, 5, 8, 10}
- `capacity_level` ∈ {tight, loose}
- `n_players` ∈ {2, 4, 6, 8}

| | **Asymmetric** (distinct s-t from layer 0 / layer L-1) |
|---|---|
| **Tight capacity** | Set E |
| **Loose capacity** | Set F |

In [ ]:
import sys, os, json
import numpy as np
from pathlib import Path
from itertools import product

# Import NFG instance generator (network generation logic is identical to DFG)
from gipg.nfg.instance import generate_nfg_2x2

SHARED_DIR = Path('../data/nfg')
SHARED_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK')

## 1. Configuration

Layered graph parameters and the experimental grid.
Only **asymmetric** symmetry is used (sources from layer 0, sinks from layer L-1).

In [ ]:
# ── Layered graph parameters ────────────────────────────────────────
GRAPH_TYPE = 'layered'
N_LAYERS_LIST = [3, 5, 8, 10]
N_WIDTH_LIST = [3, 5, 8, 10]
EDGE_PROB = 0.8

# ── Shared grid parameters ──────────────────────────────────────────
N_SEEDS = 1               # Use 50 for full experiments
N_PLAYERS_LIST = [2, 4, 6, 8]
CAPACITY_LEVELS = ['tight', 'loose']
SYMMETRIES = ['asymmetric']   # Asymmetric only for layered graphs

SET_LABELS = {
    ('tight', 'asymmetric'): 'E',
    ('loose', 'asymmetric'): 'F',
}

grid = list(product(
    CAPACITY_LEVELS, SYMMETRIES, N_PLAYERS_LIST,
    N_LAYERS_LIST, N_WIDTH_LIST, range(N_SEEDS),
))
print(f'Total parameter combinations: {len(grid)}')
print(f'Layer sizes: L ∈ {N_LAYERS_LIST}, W ∈ {N_WIDTH_LIST}, edge_prob={EDGE_PROB}')
for L in N_LAYERS_LIST:
    for W in N_WIDTH_LIST:
        print(f'  L={L}, W={W} → {L*W} nodes')

## 2. Generate and Save Networks

In [ ]:
gen_log = {label: [0, 0] for label in SET_LABELS.values()}
saved_tags = []

for cap_level, sym, n_players, n_layers, n_width, seed in grid:
    set_label = SET_LABELS[(cap_level, sym)]
    tag = f'{set_label}_{cap_level}_{sym}_n{n_players}_L{n_layers}W{n_width}_s{seed}'

    try:
        nfg_inst = generate_nfg_2x2(
            n_players=n_players,
            capacity_level=cap_level,
            symmetry=sym,
            seed=seed,
            graph_type=GRAPH_TYPE,
            n_layers=n_layers,
            n_width=n_width,
            edge_prob=EDGE_PROB,
        )
    except ValueError as e:
        gen_log[set_label][1] += 1
        print(f'  FAIL ({tag}): {e}')
        continue

    # Save network data + edge_costs (same format as 00_generate_networks)
    network_data = {
        'tag': tag,
        'n_nodes': nfg_inst.n_nodes,
        'edges': [list(e) for e in nfg_inst.edges],
        'n_players': nfg_inst.n_players,
        'sources': nfg_inst.sources.tolist(),
        'sinks': nfg_inst.sinks.tolist(),
        'demands': nfg_inst.demands.tolist(),
        'capacities': nfg_inst.capacities.tolist(),
        'edge_costs': nfg_inst.edge_costs.tolist(),
        'seed': seed,
        'meta': {
            'generator': 'shared_01_generate_layered_networks',
            'graph_type': GRAPH_TYPE,
            'n_layers': n_layers,
            'n_width': n_width,
            'edge_prob': EDGE_PROB,
            'capacity_level': cap_level,
            'symmetry': sym,
            'total_demand': int(nfg_inst.demands.sum()),
            'cap_range_actual': [int(nfg_inst.capacities.min()),
                                 int(nfg_inst.capacities.max())],
            'feasibility_bumps': nfg_inst.meta.get('feasibility_bumps', 0),
            'n_graph_nodes': nfg_inst.meta.get('n_graph_nodes', nfg_inst.n_nodes),
            'n_edges': nfg_inst.n_edges,
        },
    }

    out_path = SHARED_DIR / f'{tag}.json'
    with open(out_path, 'w') as f:
        json.dump(network_data, f, indent=2)

    gen_log[set_label][0] += 1
    saved_tags.append(tag)

print(f'\nGeneration summary:')
for label in sorted(gen_log.keys()):
    ok, fail = gen_log[label]
    print(f'  Set {label}: {ok} success, {fail} fail')
print(f'\nTotal saved: {len(saved_tags)} networks to {SHARED_DIR}/')

## 3. Sanity Check

In [ ]:
import glob

# Only check layered instances (Set E, F)
json_files = sorted(glob.glob(str(SHARED_DIR / 'E_*.json'))) + \
             sorted(glob.glob(str(SHARED_DIR / 'F_*.json')))
print(f'Total layered JSON files: {len(json_files)}')

# Check first file
if json_files:
    with open(json_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample file: {Path(json_files[0]).name}')
    print(f'  Keys: {list(sample.keys())}')
    print(f'  n_nodes={sample["n_nodes"]}, n_edges={len(sample["edges"])}')
    print(f'  n_players={sample["n_players"]}')
    print(f'  demands={sample["demands"]}')
    print(f'  sources={sample["sources"]}, sinks={sample["sinks"]}')
    print(f'  capacities range: [{min(sample["capacities"])}, {max(sample["capacities"])}]')
    print(f'  meta: {sample["meta"]}')

# Summary by set
print('\n--- By set label ---')
for label in ['E', 'F']:
    count = sum(1 for f in json_files if Path(f).name.startswith(label + '_'))
    print(f'  Set {label}: {count} instances')

# Summary by (n_layers, n_width)
print('\n--- By (n_layers, n_width) ---')
from collections import Counter
lw_counts = Counter()
for fp in json_files:
    with open(fp) as f:
        d = json.load(f)
    lw_counts[(d['meta']['n_layers'], d['meta']['n_width'])] += 1
for k in sorted(lw_counts.keys()):
    print(f'  L={k[0]}, W={k[1]} ({k[0]*k[1]} nodes): {lw_counts[k]} instances')

# Summary by n_players
print('\n--- By n_players ---')
size_counts = Counter()
for fp in json_files:
    with open(fp) as f:
        d = json.load(f)
    size_counts[d['n_players']] += 1
for k in sorted(size_counts.keys()):
    print(f'  n={k}: {size_counts[k]} instances')

## 4. Network Visualization (Layered Graphs)

One representative instance per set (E and F).

Nodes are positioned by **layer** (left → right), with within-layer nodes stacked vertically.

- **Green** nodes: source(s)
- **Red** nodes: sink(s)
- **Gray** nodes: intermediate
- Edge labels show capacity $c_e$

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

# ── Which instance to visualise? ────────────────────────────────────
VIZ_N_PLAYERS = 4
VIZ_SEED      = 0
VIZ_N_LAYERS  = 5    # pick one (L, W) pair to visualise
VIZ_N_WIDTH   = 5
# ────────────────────────────────────────────────────────────────────

def layered_layout(n_layers, n_width, edges):
    """Position nodes by their layer (x) and within-layer index (y)."""
    pos = {}
    for k in range(n_layers):
        for w in range(n_width):
            node = k * n_width + w
            x = k / max(n_layers - 1, 1)
            y = (w - (n_width - 1) / 2) / max(n_width, 1)
            pos[node] = (x, y)
    return pos


def draw_layered_network(ax, net_data, title, n_layers, n_width):
    """Draw a layered directed graph on the given axes."""
    edges = [tuple(e) for e in net_data['edges']]
    sources_set = set(net_data['sources'])
    sinks_set = set(net_data['sinks'])
    capacities = net_data['capacities']
    demands = net_data['demands']
    n_players = net_data['n_players']
    n_nodes = net_data['n_nodes']

    pos = layered_layout(n_layers, n_width, edges)

    node_radius = 0.025
    node_font = 6
    cap_font = 5

    # --- Draw edges ---
    for idx, (u, v) in enumerate(edges):
        if u not in pos or v not in pos:
            continue
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        cap = capacities[idx]

        ax.annotate(
            '', xy=(x1, y1), xytext=(x0, y0),
            arrowprops=dict(
                arrowstyle='->', color='#555555', lw=0.8,
                connectionstyle='arc3,rad=0.05',
                shrinkA=node_radius * 280, shrinkB=node_radius * 280,
            ),
        )
        mx, my = 0.5 * (x0 + x1), 0.5 * (y0 + y1)
        dx, dy = x1 - x0, y1 - y0
        length = max((dx**2 + dy**2)**0.5, 1e-6)
        ox, oy = -dy / length * 0.018, dx / length * 0.018
        ax.text(mx + ox, my + oy, str(cap), fontsize=cap_font, color='#333333',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.8))

    # --- Draw nodes ---
    for node, (x, y) in pos.items():
        is_src = node in sources_set
        is_snk = node in sinks_set
        if is_src and is_snk:
            color = '#9b59b6'
        elif is_src:
            color = '#27ae60'
        elif is_snk:
            color = '#e74c3c'
        else:
            color = '#bdc3c7'

        circle = plt.Circle((x, y), node_radius, fc=color, ec='#2c3e50', lw=1.0, zorder=5)
        ax.add_patch(circle)
        ax.text(x, y, str(node), fontsize=node_font, ha='center', va='center',
                fontweight='bold', color='white', zorder=6)

    # --- Player role annotations ---
    src_labels = defaultdict(list)
    snk_labels = defaultdict(list)
    for i, s in enumerate(net_data['sources']):
        src_labels[s].append(f's{i}')
    for i, t in enumerate(net_data['sinks']):
        snk_labels[t].append(f't{i}')

    for node, labels in src_labels.items():
        if node in pos:
            x, y = pos[node]
            ax.text(x, y + node_radius + 0.02, ','.join(labels), fontsize=5,
                    ha='center', va='bottom', color='#27ae60', fontweight='bold')
    for node, labels in snk_labels.items():
        if node in pos:
            x, y = pos[node]
            ax.text(x, y - node_radius - 0.02, ','.join(labels), fontsize=5,
                    ha='center', va='top', color='#e74c3c', fontweight='bold')

    # --- Layer boundaries ---
    for k in range(n_layers):
        x = k / max(n_layers - 1, 1)
        ax.text(x, 0.55, f'L{k}', fontsize=7, ha='center', va='bottom',
                color='#7f8c8d', fontstyle='italic')

    # --- Title and stats ---
    total_D = sum(demands)
    cap_min, cap_max = min(capacities), max(capacities)
    ax.set_title(f'{title}\n'
                 f'n={n_players}, L={n_layers}, W={n_width}, |E|={len(edges)}, '
                 f'D={total_D}, cap=[{cap_min},{cap_max}]',
                 fontsize=10, fontweight='bold')
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.6, 0.7)
    ax.set_aspect('equal')
    ax.axis('off')


# --- Pick one representative instance per set ---
viz_config = {
    'E': ('tight', 'asymmetric', 'Set E: Tight + Asymmetric (Layered)'),
    'F': ('loose', 'asymmetric', 'Set F: Loose + Asymmetric (Layered)'),
}

viz_instances = {}
for label, (cap, sym, title) in viz_config.items():
    target_tag = f'{label}_{cap}_{sym}_n{VIZ_N_PLAYERS}_L{VIZ_N_LAYERS}W{VIZ_N_WIDTH}_s{VIZ_SEED}'
    fp = SHARED_DIR / f'{target_tag}.json'
    if fp.exists():
        with open(fp) as f:
            viz_instances[label] = (json.load(f), title)
    else:
        print(f'  WARNING: {target_tag}.json not found, skipping Set {label}')

# --- Draw side by side ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f'Layered Graph Instances  '
             f'(n={VIZ_N_PLAYERS}, L={VIZ_N_LAYERS}, W={VIZ_N_WIDTH}, seed={VIZ_SEED})',
             fontsize=14, fontweight='bold', y=1.0)

for i, label in enumerate(['E', 'F']):
    ax = axes[i]
    if label in viz_instances:
        net_data, title = viz_instances[label]
        draw_layered_network(ax, net_data, title, VIZ_N_LAYERS, VIZ_N_WIDTH)
    else:
        ax.text(0.5, 0.5, f'Set {label}\n(not generated)', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='gray')
        ax.axis('off')

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#27ae60', edgecolor='#2c3e50', label='Source node (layer 0)'),
    mpatches.Patch(facecolor='#e74c3c', edgecolor='#2c3e50', label='Sink node (layer L-1)'),
    mpatches.Patch(facecolor='#bdc3c7', edgecolor='#2c3e50', label='Intermediate node'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=9, frameon=True, fancybox=True)

plt.tight_layout(rect=[0, 0.06, 1, 0.96])
out_name = f'network_layered_n{VIZ_N_PLAYERS}_L{VIZ_N_LAYERS}W{VIZ_N_WIDTH}_s{VIZ_SEED}.png'
plt.savefig(str(SHARED_DIR / out_name), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {SHARED_DIR / out_name}')